# APRA Talk — Sample Dataset Generator

This notebook generates six CSV files that mimic a major gifts portfolio pulled from a fundraising CRM:

| File | Description |
|------|-------------|
| `fundraisers.csv` | The 5 Major Gifts Officers |
| `constituents.csv` | Householded prospect records (one row per household) |
| `actions.csv` | All contact reports — multiple rows per constituent |
| `gifts.csv` | All gifts — multiple rows per constituent |
| `proposals.csv` | Formal solicitation proposals with ask and funded amounts |
| `ratings.csv` | Wealth screening ratings — 2–3 per constituent |

The tables are **intentionally separate** so we can demonstrate joining them in Tableau.
`fundraiser_name` does not appear in any table other than `fundraisers.csv` — join there to get it.

---
### Things you can tweak in Cell 1:
- `PROSPECTS_PER_FUNDRAISER` — how many people each MGO has
- `SEED` — change this to get a different random dataset
- Names, territories, emails of the fundraisers
- Status weights, capacity rating weights

In [ ]:
import random
import csv
import os
from datetime import date, timedelta

# ── Configuration ─────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)

TODAY = date(2026, 5, 22)
OUTPUT_DIR = os.path.dirname(os.path.abspath('__file__'))  # same folder as this notebook

# How many prospects each fundraiser manages
PROSPECTS_PER_FUNDRAISER = [85, 92, 78, 88, 75]

# ── Fundraisers ───────────────────────────────────────────────────────────────
FUNDRAISERS = [
    {"fundraiser_id": 1, "name": "Sarah Chen",       "title": "Senior Major Gifts Officer", "territory": "Northeast",    "email": "s.chen@university.edu"},
    {"fundraiser_id": 2, "name": "Marcus Williams",  "title": "Major Gifts Officer",        "territory": "Southeast",    "email": "m.williams@university.edu"},
    {"fundraiser_id": 3, "name": "Priya Patel",      "title": "Senior Major Gifts Officer", "territory": "Midwest",      "email": "p.patel@university.edu"},
    {"fundraiser_id": 4, "name": "James O'Brien",    "title": "Major Gifts Officer",        "territory": "West",         "email": "j.obrien@university.edu"},
    {"fundraiser_id": 5, "name": "Linda Okafor",     "title": "Major Gifts Officer",        "territory": "International", "email": "l.okafor@university.edu"},
]

print(f"Fundraisers defined: {len(FUNDRAISERS)}")

In [ ]:
# ── Name pool ──────────────────────────────────────────────────────────────────
# Realistic mix of first names and last names for a university donor base

FIRST_NAMES = [
    "James", "Mary", "Robert", "Patricia", "John", "Jennifer", "Michael", "Linda",
    "William", "Barbara", "David", "Susan", "Richard", "Jessica", "Joseph", "Sarah",
    "Thomas", "Karen", "Charles", "Lisa", "Christopher", "Nancy", "Daniel", "Betty",
    "Matthew", "Dorothy", "Anthony", "Sandra", "Mark", "Ashley", "Donald", "Margaret",
    "Steven", "Kimberly", "Paul", "Emily", "Andrew", "Donna", "Kenneth", "Michelle",
    "George", "Carol", "Joshua", "Amanda", "Kevin", "Melissa", "Brian", "Deborah",
    "Edward", "Stephanie", "Ronald", "Rebecca", "Timothy", "Sharon", "Jason", "Laura",
    "Jeffrey", "Cynthia", "Ryan", "Kathleen", "Gary", "Amy", "Jacob", "Angela",
    "Nicholas", "Shirley", "Eric", "Anna", "Jonathan", "Brenda", "Stephen", "Pamela",
    "Larry", "Emma", "Justin", "Nicole", "Scott", "Helen", "Brandon", "Samantha",
    "Benjamin", "Katherine", "Samuel", "Christine", "Frank", "Debra", "Raymond", "Rachel",
    "Gregory", "Carolyn", "Frank", "Janet", "Alexander", "Catherine", "Patrick", "Maria",
    "Jack", "Heather", "Dennis", "Diane", "Jerry", "Julie", "Tyler", "Joyce",
    "Aaron", "Victoria", "Henry", "Kelly", "Jose", "Christina", "Adam", "Ruth",
    "Douglas", "Joan", "Peter", "Virginia", "Nathan", "Judith", "Zachary", "Evelyn",
    "Walter", "Jean", "Kyle", "Frances", "Harold", "Cheryl", "Carl", "Alice",
    "Arthur", "Ann", "Gerald", "Jean", "Roger", "Marie",
    # Some less common names for variety
    "Eston", "Thaddeus", "Cornelius", "Prescott", "Harrington", "Whitfield",
    "Reginald", "Beaumont", "Montgomery", "Ellsworth", "Barnard", "Clifton",
    "Miriam", "Constance", "Eugenia", "Harriet", "Marguerite", "Rosalind",
]

LAST_NAMES = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller", "Davis",
    "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez", "Wilson", "Anderson",
    "Thomas", "Taylor", "Moore", "Jackson", "Martin", "Lee", "Perez", "Thompson",
    "White", "Harris", "Sanchez", "Clark", "Ramirez", "Lewis", "Robinson", "Walker",
    "Young", "Allen", "King", "Wright", "Scott", "Torres", "Nguyen", "Hill",
    "Flores", "Green", "Adams", "Nelson", "Baker", "Hall", "Rivera", "Campbell",
    "Mitchell", "Carter", "Roberts", "Turner", "Phillips", "Evans", "Collins",
    "Stewart", "Morales", "Murphy", "Cook", "Rogers", "Gutierrez", "Ortiz",
    "Morgan", "Cooper", "Peterson", "Bailey", "Reed", "Kelly", "Howard", "Ramos",
    "Kim", "Cox", "Ward", "Richardson", "Watson", "Brooks", "Chavez", "Wood",
    "James", "Bennett", "Gray", "Mendoza", "Ruiz", "Hughes", "Price", "Alvarez",
    "Castillo", "Sanders", "Patel", "Myers", "Long", "Ross", "Foster", "Jimenez",
    # More distinctive last names for major donor feel
    "Whitfield", "Harrington", "Worthington", "Ashworth", "Blackwell", "Sutton",
    "Pemberton", "Ashford", "Langford", "Kingsford", "Wentworth", "Fairbanks",
    "Holloway", "Thornton", "Whitmore", "Aldridge", "Bancroft", "Beaumont",
    "Chen", "Park", "Nguyen", "Zhang", "Okafor", "Mensah", "Diallo", "Abubakar",
]

MIDDLE_INITIALS = list("ABCDEFGHJKLMNPRSTW")  # common middle initials
PREFIXES = ["Dr.", "Mr.", "Mrs.", "Ms.", "Prof.", "", "", "", ""]  # weighted toward no prefix
SUFFIXES = ["", "", "", "", "", "Jr.", "Sr.", "III", "II"]  # weighted toward no suffix

def make_name():
    """Generate a formal constituent name like RE would store."""
    prefix = random.choice(PREFIXES)
    first = random.choice(FIRST_NAMES)
    middle = random.choice(MIDDLE_INITIALS) if random.random() < 0.5 else None
    last = random.choice(LAST_NAMES)
    suffix = random.choice(SUFFIXES)
    parts = [p for p in [prefix, first, f"{middle}." if middle else None, last, suffix] if p]
    return " ".join(parts)

def make_birthdate(min_year=1940, max_year=1985):
    year = random.randint(min_year, max_year)
    month = random.randint(1, 12)
    day = random.randint(1, 28)
    return date(year, month, day)

print("Name pool ready. Sample names:")
for _ in range(5):
    print(f"  {make_name()}")

In [ ]:
# ── Constituent data pools ─────────────────────────────────────────────────────

STATUSES = ["Identification", "Qualification", "Cultivation", "Solicitation", "Stewardship", "Disqualified"]

# Probability weights for each status (must sum to 1)
STATUS_WEIGHTS = [0.12, 0.18, 0.38, 0.15, 0.12, 0.05]

# Capacity ratings — what the prospect research team thinks they can give
CAPACITY_RATINGS = ["$50K–$99K", "$100K–$249K", "$250K–$499K", "$500K–$999K", "$1M+"]
CAPACITY_WEIGHTS = [0.35, 0.30, 0.18, 0.12, 0.05]  # most prospects are at the lower end

# Typical dollar ranges per capacity rating (for generating gift amounts)
CAPACITY_GIFT_RANGES = {
    "$50K–$99K":    (5_000,   75_000),
    "$100K–$249K":  (10_000, 200_000),
    "$250K–$499K":  (25_000, 400_000),
    "$500K–$999K":  (50_000, 800_000),
    "$1M+":         (100_000, 2_000_000),
}

# How many days they've been in their current status (varies by stage)
DAYS_IN_STATUS_RANGES = {
    "Identification": (7, 180),
    "Qualification":  (14, 365),
    "Cultivation":    (90, 730),
    "Solicitation":   (30, 400),
    "Stewardship":    (60, 1095),
    "Disqualified":   (14, 500),
}

# Cities and states (skewed toward university alumni hubs)
LOCATIONS = [
    ("New York", "NY"), ("Boston", "MA"), ("Chicago", "IL"), ("San Francisco", "CA"),
    ("Los Angeles", "CA"), ("Houston", "TX"), ("Philadelphia", "PA"), ("Atlanta", "GA"),
    ("Washington", "DC"), ("Seattle", "WA"), ("Denver", "CO"), ("Minneapolis", "MN"),
    ("Dallas", "TX"), "Miami", ("Charlotte", "NC"), ("Phoenix", "AZ"),
    ("Detroit", "MI"), ("Portland", "OR"), ("Nashville", "TN"), ("Raleigh", "NC"),
    ("Austin", "TX"), ("Baltimore", "MD"), ("Pittsburgh", "PA"), ("St. Louis", "MO"),
    ("San Diego", "CA"), ("Columbus", "OH"), ("Indianapolis", "IN"), ("Louisville", "KY"),
    ("Richmond", "VA"), ("Hartford", "CT"), ("Providence", "RI"), ("Albany", "NY"),
    # International (for Linda's territory)
    ("London", "UK"), ("Toronto", "Canada"), ("Sydney", "Australia"), ("Hong Kong", "HK"),
    ("Singapore", "SG"), ("Dubai", "UAE"),
]
# Fix the one entry that's a string instead of tuple
LOCATIONS = [loc if isinstance(loc, tuple) else (loc, "FL") for loc in LOCATIONS]

# Class years (graduation year — this is a university context)
CLASS_YEARS = list(range(1962, 2010))

print("Data pools ready.")

In [ ]:
# ── Generate Constituents ──────────────────────────────────────────────────────

constituents = []
constituent_id = 1

for fundraiser in FUNDRAISERS:
    count = PROSPECTS_PER_FUNDRAISER[fundraiser["fundraiser_id"] - 1]

    for _ in range(count):
        status = random.choices(STATUSES, weights=STATUS_WEIGHTS)[0]
        days_range = DAYS_IN_STATUS_RANGES[status]
        capacity = random.choices(CAPACITY_RATINGS, weights=CAPACITY_WEIGHTS)[0]
        city, state = random.choice(LOCATIONS)

        has_spouse = random.random() < 0.42
        spouse_name = make_name() if has_spouse else ""
        spouse_birthdate = str(make_birthdate()) if has_spouse else ""

        class_year = random.choice(CLASS_YEARS) if random.random() < 0.80 else ""

        constituents.append({
            "constituent_id":   constituent_id,
            "name":             make_name(),
            "birthdate":        str(make_birthdate()),
            "status":           status,
            "days_in_status":   random.randint(*days_range),
            "spouse_name":      spouse_name,
            "spouse_birthdate": spouse_birthdate,
            "fundraiser_id":    fundraiser["fundraiser_id"],  # join to fundraisers.csv for name
            "capacity_rating":  capacity,
            "city":             city,
            "state":            state,
            "class_year":       class_year,
        })
        constituent_id += 1

print(f"Generated {len(constituents)} constituents")
print("\nStatus breakdown:")
from collections import Counter
for status, cnt in sorted(Counter(c["status"] for c in constituents).items()):
    print(f"  {status:16s} {cnt}")

In [ ]:
# ── Generate Actions ──────────────────────────────────────────────────────────
#
# action_number: 1 = most recent. All rows kept — not pre-filtered.
# "Portfolio Review" replaces "Proposal Submitted" to avoid conceptual
# overlap with the proposals table.

ACTION_COUNT_RANGES = {
    "Identification": (1, 3),
    "Qualification":  (2, 6),
    "Cultivation":    (5, 15),
    "Solicitation":   (8, 22),
    "Stewardship":    (3, 12),
    "Disqualified":   (1, 5),
}

CATEGORIES = ["Meeting", "Phone Call", "Email", "Site Visit", "Event Attendance", "Portfolio Review"]

SUMMARIES = {
    "Meeting": [
        "Coffee meeting downtown", "Lunch at faculty club", "Office visit",
        "Met at alumni event", "Discovery meeting", "Stewardship dinner",
        "Discussed naming opportunity", "Initial in-person meeting",
        "Follow-up meeting re: proposal", "Met spouse for first time",
        "Campus visit scheduled", "Year-end thank-you lunch",
    ],
    "Phone Call": [
        "Check-in call", "Solicitation follow-up", "Discussed pledge payment",
        "Left voicemail", "Call re: event invitation", "Introduced new initiative",
        "Confirmed meeting time", "Touched base on family news",
        "Called to share impact story", "Brief check-in",
    ],
    "Email": [
        "Sent impact report", "Event invitation", "Holiday greeting",
        "Shared news article of interest", "Proposal follow-up email",
        "Sent thank-you note", "Updated on campaign progress",
        "Forwarded dean's letter", "Sent birthday greeting",
        "Annual fund solicitation",
    ],
    "Site Visit": [
        "Toured new building", "Lab walkthrough with faculty",
        "Met scholarship recipient", "Campus tour with family",
        "Athletic facility tour", "Arts center opening",
        "Visited endowed professorship",
    ],
    "Event Attendance": [
        "Attended gala", "Homecoming reception", "President's dinner",
        "Alumni leadership weekend", "Regional club event",
        "Advisory board meeting", "Commencement ceremony",
        "Athletic event in donor suite", "Campaign kickoff event",
    ],
    "Portfolio Review": [
        "Debrief after proposal conversation", "Internal portfolio review",
        "Reviewed prospect strategy with team", "Pre-ask strategy session",
        "Discussed ask timing with director", "Solicitation debrief",
        "Strategy call with leadership",
    ],
}

actions = []
action_id = 1

for c in constituents:
    status = c["status"]
    n_actions = random.randint(*ACTION_COUNT_RANGES[status])
    action_dates = sorted(
        [TODAY - timedelta(days=random.randint(1, 1460)) for _ in range(n_actions)],
        reverse=True
    )
    for i, action_date in enumerate(action_dates):
        category = random.choice(CATEGORIES)
        actions.append({
            "action_id":      action_id,
            "constituent_id": c["constituent_id"],
            "action_number":  i + 1,
            "category":       category,
            "action_date":    str(action_date),
            "summary":        random.choice(SUMMARIES[category]),
            "fundraiser_id":  c["fundraiser_id"],
        })
        action_id += 1

print(f"Generated {len(actions)} actions")
print(f"Average per constituent: {len(actions)/len(constituents):.1f}")

In [ ]:
# ── Generate Gifts ────────────────────────────────────────────────────────────
#
# gift_number: 1 = most recent. All rows kept.
# Gift types: Donation and Pledge only — appropriate for individual major donors.

GIFT_COUNT_RANGES = {
    "Identification":  (0, 1),
    "Qualification":   (0, 2),
    "Cultivation":     (0, 4),
    "Solicitation":    (1, 6),
    "Stewardship":     (2, 10),
    "Disqualified":    (0, 3),
}

GIFT_PROBABILITY = {
    "Identification": 0.25,
    "Qualification":  0.45,
    "Cultivation":    0.60,
    "Solicitation":   0.85,
    "Stewardship":    1.00,
    "Disqualified":   0.40,
}

GIFT_TYPES        = ["Donation", "Pledge"]
GIFT_TYPE_WEIGHTS = [0.70, 0.30]

GIFT_SUBTYPES = {
    "Donation": ["", "", "", "GiveCampus", "Wire Transfer", "Check"],
    "Pledge":   ["", "Multi-year Pledge", "Campaign Pledge"],
}

DESIGNATIONS = [
    "Annual Fund", "Scholarship Endowment", "Capital Campaign",
    "Faculty Excellence Fund", "Athletics Fund", "Arts & Humanities",
    "Research Initiative", "Library Fund", "Student Life", "Global Initiatives",
    "Unrestricted Endowment", "Dean's Discretionary Fund",
]

gifts = []
gift_id = 1

for c in constituents:
    status = c["status"]
    capacity = c["capacity_rating"]
    gift_range = CAPACITY_GIFT_RANGES[capacity]

    if random.random() > GIFT_PROBABILITY[status]:
        continue

    n_gifts = random.randint(*GIFT_COUNT_RANGES[status])
    if n_gifts == 0:
        continue

    gift_dates = sorted(
        [TODAY - timedelta(days=random.randint(30, 5475)) for _ in range(n_gifts)],
        reverse=True
    )

    for i, gift_date in enumerate(gift_dates):
        gift_type = random.choices(GIFT_TYPES, weights=GIFT_TYPE_WEIGHTS)[0]
        raw_amount = random.uniform(*gift_range)
        if raw_amount >= 50_000:   amount = round(raw_amount / 10_000) * 10_000
        elif raw_amount >= 10_000: amount = round(raw_amount / 5_000) * 5_000
        else:                      amount = round(raw_amount / 1_000) * 1_000
        amount = max(amount, 5_000)

        if gift_type == "Pledge":
            days_old = (TODAY - gift_date).days
            if days_old < 1095:
                pct_paid = min(days_old / 1095, 1.0) * random.uniform(0.3, 0.9)
                gift_balance = round(amount * (1 - pct_paid) / 1000) * 1000
            else:
                gift_balance = 0.0
        else:
            gift_balance = 0.0

        gifts.append({
            "gift_id":        gift_id,
            "constituent_id": c["constituent_id"],
            "gift_number":    i + 1,
            "amount":         float(amount),
            "gift_date":      str(gift_date),
            "gift_type":      gift_type,
            "gift_subtype":   random.choice(GIFT_SUBTYPES[gift_type]),
            "gift_balance":   float(gift_balance),
            "designation":    random.choice(DESIGNATIONS),
        })
        gift_id += 1

print(f"Generated {len(gifts)} gifts")
print(f"Gift types: {dict(Counter(g['gift_type'] for g in gifts))}")
print(f"Total giving: ${sum(g['amount'] for g in gifts):,.0f}")

In [ ]:
# ── Write CSV files ────────────────────────────────────────────────────────────

def write_csv(filename, rows, fieldnames):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Wrote {len(rows):>5,} rows → {filename}")

print("Writing CSV files...")

write_csv(
    "fundraisers.csv",
    FUNDRAISERS,
    ["fundraiser_id", "name", "title", "territory", "email"]
)

write_csv(
    "constituents.csv",
    constituents,
    ["constituent_id", "name", "birthdate", "status", "days_in_status",
     "spouse_name", "spouse_birthdate", "fundraiser_id",
     "capacity_rating", "city", "state", "class_year"]
)

write_csv(
    "actions.csv",
    actions,
    ["action_id", "constituent_id", "action_number", "category",
     "action_date", "summary", "fundraiser_id"]
)

write_csv(
    "gifts.csv",
    gifts,
    ["gift_id", "constituent_id", "gift_number", "amount",
     "gift_date", "gift_type", "gift_subtype", "gift_balance", "designation"]
)

write_csv(
    "proposals.csv",
    proposals,
    ["proposal_id", "constituent_id", "fundraiser_id",
     "proposal_date", "ask_amount", "funded_amount", "status",
     "designation", "expected_close_date"]
)

write_csv(
    "ratings.csv",
    ratings,
    ["rating_id", "constituent_id", "category", "source", "value", "value_type", "date"]
)

print("\nDone! All files written to:", OUTPUT_DIR)

In [ ]:
# ── Generate Ratings ──────────────────────────────────────────────────────────
#
# 2–3 ratings per constituent across three categories.
# value is stored as a plain number — use value_type to know the unit.
#   Currency → format as dollars in Tableau
#   Percent  → format as percentage in Tableau

RATING_SOURCES = ["DonorSearch", "iWave", "Blackbaud Target Analytics", "WealthEngine"]

CAPACITY_TO_DOLLARS = {
    "$50K-$99K":   (40_000,   120_000),
    "$100K-$249K": (80_000,   300_000),
    "$250K-$499K": (200_000,  600_000),
    "$500K-$999K": (400_000, 1_200_000),
    "$1M+":        (800_000, 5_000_000),
}

LIKELIHOOD_RANGES = {
    "Identification": (10, 40),
    "Qualification":  (20, 55),
    "Cultivation":    (40, 75),
    "Solicitation":   (55, 90),
    "Stewardship":    (65, 95),
    "Disqualified":   (5,  30),
}

def round_dollars(amount, nearest=10_000):
    return round(amount / nearest) * nearest

ratings = []
rating_id = 1

for c in constituents:
    cap    = c["capacity_rating"]
    source = random.choice(RATING_SOURCES)
    cats   = random.sample(["Estimated Capacity", "Likelihood to Give", "Real Estate"], k=random.randint(2, 3))

    for category in cats:
        rated_on = TODAY - timedelta(days=random.randint(30, 730))

        if category == "Estimated Capacity":
            lo, hi = CAPACITY_TO_DOLLARS.get(cap, (50_000, 300_000))
            value      = float(round_dollars(random.uniform(lo, hi)))
            value_type = "Currency"

        elif category == "Likelihood to Give":
            lo, hi = LIKELIHOOD_RANGES.get(c["status"], (20, 60))
            value      = float(random.randint(lo, hi))
            value_type = "Percent"

        else:  # Real Estate
            value      = float(round_dollars(random.uniform(200_000, 4_000_000), nearest=50_000))
            value_type = "Currency"

        ratings.append({
            "rating_id":      rating_id,
            "constituent_id": c["constituent_id"],
            "category":       category,
            "source":         source,
            "value":          value,
            "value_type":     value_type,
            "date":           str(rated_on),
        })
        rating_id += 1

print(f"Generated {len(ratings)} ratings")
print("\nCategory breakdown:")
for cat, n in sorted(Counter(r["category"] for r in ratings).items()):
    print(f"  {cat:<25} {n}")

In [ ]:
# ── Write CSV files ────────────────────────────────────────────────────────────

def write_csv(filename, rows, fieldnames):
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    print(f"  Wrote {len(rows):>5,} rows → {filename}")

print("Writing CSV files...")

write_csv(
    "fundraisers.csv",
    FUNDRAISERS,
    ["fundraiser_id", "name", "title", "territory", "email"]
)

write_csv(
    "constituents.csv",
    constituents,
    ["constituent_id", "name", "birthdate", "status", "days_in_status",
     "spouse_name", "spouse_birthdate", "fundraiser_id", "fundraiser_name",
     "capacity_rating", "city", "state", "class_year"]
)

write_csv(
    "actions.csv",
    actions,
    ["action_id", "constituent_id", "action_number", "category",
     "action_date", "summary", "fundraiser_id", "fundraiser_name", "days_since"]
)

write_csv(
    "gifts.csv",
    gifts,
    ["gift_id", "constituent_id", "gift_number", "amount",
     "gift_date", "gift_type", "gift_subtype", "gift_balance", "designation"]
)

print("\nDone! All files written to:", OUTPUT_DIR)